# Sesión 08 - Lab 2: Un mismo Job con file arrival trigger y table update trigger

Un Job de Lakeflow puede tener más de un trigger configurado a la vez. Este lab arma **un único Job** con dos triggers en paralelo: uno por llegada de archivo (`File arrival`) y otro por actualización de tabla (`Table update`). Compara qué dispara cada uno mirando la pestaña **Runs** del Job.

La tarea que orquesta ambos triggers es siempre la misma (Lab 2B): registra una fila de auditoría con la hora de ejecución. Lo que cambia es *qué evento la disparó*, visible en la columna **Trigger** de cada corrida.

## Verificación del entorno

In [ ]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_08")

## Lab 2A — Setup: tabla y carpeta a monitorear

Crea la tabla `dbassociate.default.pedidos_monitor_trigger` (la que va a vigilar el table update trigger) con un par de filas semilla, y la subcarpeta `sesion_08/triggers/` del Volume (la que va a vigilar el file arrival trigger).

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import current_timestamp, lit

schema_monitor = StructType([
    StructField("pedido_id", StringType(), False),
    StructField("canal", StringType(), True),
    StructField("monto_total", DoubleType(), True),
])

df_seed = spark.createDataFrame(
    [("PED-SEED-01", "web", 120.0), ("PED-SEED-02", "app", 45.5)],
    schema=schema_monitor,
)

df_seed = df_seed.withColumn("registrado_en", current_timestamp())

df_seed.write.mode("overwrite").saveAsTable("dbassociate.default.pedidos_monitor_trigger")

dbutils.fs.mkdirs("/Volumes/dbassociate/default/vol_landing/sesion_08/triggers")

print("Tabla semilla y carpeta de triggers listas.")
display(spark.table("dbassociate.default.pedidos_monitor_trigger"))

## Lab 2B — TAREA del Job: registrar auditoría de ejecución

Esta es la única tarea del Job (`sesion08_job_triggers`, ver Lab 2C): sin importar qué trigger la haya disparado, deja una fila en `dbassociate.default.log_ejecuciones_orquestacion` con la hora de la corrida. Es el mismo notebook que se apunta como `Notebook task` al crear el Job.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

spark.sql("""
    CREATE TABLE IF NOT EXISTS dbassociate.default.log_ejecuciones_orquestacion (
        ejecutado_en TIMESTAMP,
        pedidos_en_tabla_monitor BIGINT
    )
""")

total_pedidos = spark.table("dbassociate.default.pedidos_monitor_trigger").count()

df_log = spark.createDataFrame(
    [(total_pedidos,)], schema=["pedidos_en_tabla_monitor"]
).withColumn("ejecutado_en", current_timestamp())

df_log.select("ejecutado_en", "pedidos_en_tabla_monitor").write.mode("append").saveAsTable(
    "dbassociate.default.log_ejecuciones_orquestacion"
)

print(f"Ejecución registrada. Filas en la tabla monitoreada: {total_pedidos}")

## Lab 2C — Crear el Job con los dos triggers

**Workflows → Jobs & Pipelines → Create Job.** Nombre: `sesion08_job_triggers`.

**Tarea única — `registrar_auditoria`**
- Type: `Notebook` · Source: `Workspace`, este notebook (`sesion08_lab2.ipynb`)
- Compute: Serverless

**Trigger 1 — File arrival** (Job details → Schedules & Triggers → Add trigger)
- Trigger type: `File arrival`
- Storage location: `/Volumes/dbassociate/default/vol_landing/sesion_08/triggers/`
- Advanced → **Wait after last change in seconds**: `60` (agrupa varios archivos que lleguen juntos en una sola corrida en vez de una por archivo)
- **Test connection** → **Save**

**Trigger 2 — Table update** (Add trigger otra vez, sobre el mismo Job)
- Trigger type: `Table update`
- Tables: `dbassociate.default.pedidos_monitor_trigger`
- Trigger when: `Any table is updated` (acá solo hay una tabla, pero esta opción importa si se monitorean varias)
- Advanced → **Minimum time between triggers in seconds**: `30`
- **Test trigger** → **Save**

El Job queda con **2 triggers activos** al mismo tiempo: la UI lo muestra como `Multiple` en la columna de tipo de trigger de la lista de Jobs.

## Lab 2D — Disparar el file arrival trigger

Subí el archivo `pedidos_lote_trigger.csv` (en la carpeta de esta sesión, junto a este notebook) a `/Volumes/dbassociate/default/vol_landing/sesion_08/triggers/` desde **Catalog Explorer** (arrastrar y soltar). Unos segundos después de que termine de subir (más los 60 segundos de `Wait after last change` configurados en Lab 2C), el Job dispara una corrida. Confirmá en la pestaña **Runs** que la columna **Trigger** dice `File arrival`.

In [ ]:
# Verificación: confirmar que el archivo llegó a la carpeta monitoreada
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_08/triggers")

## Lab 2E — Disparar el table update trigger

Esta celda simula una tarea upstream (por ejemplo, otro Job de ingesta) escribiendo nuevas filas en la tabla monitoreada. Al confirmar el `INSERT`, Unity Catalog registra el commit y el table update trigger dispara una corrida. Confirmá en **Runs** que esta vez la columna **Trigger** dice `Table update`.

In [ ]:
spark.sql("""
    INSERT INTO dbassociate.default.pedidos_monitor_trigger
    VALUES ('PED-SEED-03', 'tienda', 210.0, current_timestamp())
""")

print("Insert confirmado en la tabla monitoreada.")

## Lab 2F — Comparar comportamiento

| | File arrival | Table update |
|---|---|---|
| Qué vigila | Una ruta de Volume/external location en Unity Catalog | Una o más tablas de Unity Catalog |
| Requiere | `READ` sobre el storage location | `CAN MANAGE` sobre el Job, tabla en UC |
| Agrupa eventos con | `Wait after last change in seconds` | `Minimum time between triggers` + `Wait after last change` |
| Con varias fuentes | Un único path (puede tener subcarpetas) | Elegís `All tables updated` o `Any table updated` |
| Valores dinámicos disponibles | N/A | `{{job.trigger.table_update.updated_tables}}`, `commit_timestamp`, `version` |

**Concurrencia:** por default un Job solo admite **una corrida activa a la vez** (`Maximum concurrent runs = 1`). Si el file arrival trigger y el table update trigger se disparan casi al mismo tiempo, la segunda corrida queda en estado `Queued` hasta que la primera termina: no se pierde, pero tampoco corre en paralelo salvo que subas ese límite en **Advanced → Edit concurrent runs**.

**Fallos silenciosos:** ni el file arrival ni el table update trigger avisan solos si el trigger en sí falla al evaluarse (por ejemplo, si se revocan permisos sobre la tabla o el Volume). Para enterarte, hay que configurar una notificación de **on failure** sobre el Job (Job details → Notifications → Add), no solo confiar en que "si no llegó ningún run, es que no pasó nada".

## Limpieza

In [ ]:
# spark.sql("DROP TABLE IF EXISTS dbassociate.default.log_ejecuciones_orquestacion")
# spark.sql("DROP TABLE IF EXISTS dbassociate.default.pedidos_monitor_trigger")
# dbutils.fs.rm("/Volumes/dbassociate/default/vol_landing/sesion_08/triggers", recurse=True)

# print("Tablas y carpeta de este laboratorio eliminadas.")